# PGx Risk Model Training & Analysis Pipeline

This notebook orchestrates **risk model training and analysis steps (Steps 6-8)** of the PGx analysis pipeline with **status tracking** and **parallel execution** capabilities.

## Prerequisites

⚠️ **Feature Engineering Must Be Complete**

Before running this notebook, ensure all feature engineering steps (Steps 3-5d) are complete:
- Use `pgx_cohort_feature_engineering.ipynb` to complete feature engineering
- Review and validate feature engineering results before proceeding
- All feature files must exist in `5_feature_engineering/feature_engineering_outputs/` or step-specific output directories

## Risk Model Workflow

1. **Step 6: Final Model Training** (`6b_final_model_selection/`)
   - Assembles final feature table from all feature engineering outputs
   - Trains CatBoost, XGBoost, XGBoost RF with Monte Carlo Cross-Validation
   - Model selection and evaluation

2. **Step 7: FFA Analysis** (`7_ffa_analysis/`)
   - Formal Feature Attribution, causal analysis
   - Explains model predictions and identifies causal relationships

3. **Step 8: SHAP Analysis** (`8_shap_analysis/`)
   - SHAP values, feature importance, prediction explanations
   - Post-model analysis for feature importance

## Notebook Features

- ✅ **Pipeline Orchestration** – Run full risk model pipeline or individual steps
- ✅ **Status Tracking** – Check completion status of each step per cohort/age band
- ✅ **Artifact Verification** – Automatically detect completed steps
- ✅ **Parallel Execution** – Run multiple cohorts in parallel using ProcessPoolExecutor
- ✅ **Dependency Checking** – Verifies feature engineering is complete before model training

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
from typing import Dict, List, Tuple, Optional
from datetime import datetime

def resolve_project_root() -> Path:
    """Resolve the pgx-analysis project root for both notebook and script modes."""
    # When run as a script, __file__ is defined
    if "__file__" in globals():
        return Path(__file__).resolve().parents[0]

    # When run inside Jupyter, fall back to current working directory and parents
    cwd = Path(os.getcwd()).resolve()
    # Check for risk model notebook first
    if (cwd / "pgx_risk_model.ipynb").exists():
        return cwd
    
    # Check for original pipeline notebook
    if (cwd / "pgx_cohort_pipeline.ipynb").exists():
        return cwd

    for parent in cwd.parents:
        if (parent / "pgx_risk_model.ipynb").exists() or (parent / "pgx_cohort_pipeline.ipynb").exists():
            return parent

    return cwd


PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import constants
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

# ------------------------------------------------------------------
# Cohort configuration for risk model training
# ------------------------------------------------------------------
COHORT_NAME = "opioid_ed"       # or "non_opioid_ed"
AGE_BAND = "25-44"              # e.g., "0-12", "13-24", ..., "85-94"
TRAIN_YEARS = [2016, 2017, 2018]
TEST_YEAR = 2019

print(
    f"[CONFIG] cohort={COHORT_NAME}, age_band={AGE_BAND}, "
    f"train_years={TRAIN_YEARS}, test_year={TEST_YEAR}"
)

## Risk Model Status Tracking

Functions to check completion status of risk model steps and verify feature engineering prerequisites.

In [ ]:
def check_feature_engineering_prerequisites(cohort_name: str, age_band: str) -> Dict[str, bool]:
    """
    Check if all feature engineering prerequisites are complete before model training.
    
    Returns a dictionary indicating which feature files exist.
    """
    age_band_fname = age_band.replace("-", "_")
    prerequisites = {}
    
    # Check for feature files in primary location first, then fallback
    feature_base = PROJECT_ROOT / "5_feature_engineering" / "feature_engineering_outputs"
    
    # BupaR features
    bupar_path = feature_base / "5_bupar" / cohort_name / age_band / f"bupaR_added_features_{cohort_name}_{age_band_fname}.csv"
    if not bupar_path.exists():
        bupar_path = PROJECT_ROOT / "5a_bupaR_analysis" / "outputs" / "feature_engineering" / f"bupaR_added_features_{cohort_name}_{age_band_fname}.csv"
    prerequisites["bupar"] = bupar_path.exists()
    
    # FP-Growth features
    fpgrowth_path = feature_base / "4_fpgrowth" / cohort_name / age_band / f"fpgrowth_added_features_{cohort_name}_{age_band_fname}.csv"
    if not fpgrowth_path.exists():
        fpgrowth_path = PROJECT_ROOT / "5b_fpgrowth_analysis" / "outputs" / "feature_engineering" / f"fpgrowth_added_features_{cohort_name}_{age_band_fname}.csv"
    prerequisites["fpgrowth"] = fpgrowth_path.exists()
    
    # PGx features
    pgx_path = feature_base / "7_pgx" / cohort_name / age_band / f"pgx_added_features_{cohort_name}_{age_band_fname}.csv"
    if not pgx_path.exists():
        pgx_path = PROJECT_ROOT / "5c_pgx_analysis" / "outputs" / "feature_engineering" / f"pgx_added_features_{cohort_name}_{age_band_fname}.csv"
    prerequisites["pgx"] = pgx_path.exists()
    
    # DTW features
    dtw_path = feature_base / "6_dtw" / cohort_name / age_band / f"dtw_added_features_{cohort_name}_{age_band_fname}.csv"
    if not dtw_path.exists():
        dtw_path = PROJECT_ROOT / "5d_dtw_analysis" / "outputs" / "feature_engineering" / f"dtw_added_features_{cohort_name}_{age_band_fname}.csv"
    prerequisites["dtw"] = dtw_path.exists()
    
    # Model data (required for final feature assembly)
    model_data_path = PROJECT_ROOT / "4a_model_data" / f"cohort_name={cohort_name}" / f"age_band={age_band}" / "model_events_no_protocols.parquet"
    if not model_data_path.exists():
        model_data_path = PROJECT_ROOT / "4a_model_data" / f"cohort_name={cohort_name}" / f"age_band={age_band}" / "model_events.parquet"
    prerequisites["model_data"] = model_data_path.exists()
    
    return prerequisites


def check_risk_model_status(cohort_name: str, age_band: str) -> Dict[str, str]:
    """
    Check completion status of all risk model steps (Steps 6-8) for a given cohort/age band.
    
    Returns a dictionary mapping step names to status: 'DONE', 'PENDING', 'PARTIAL', or 'ERROR'
    """
    age_band_fname = age_band.replace("-", "_")
    status = {}
    
    # Step 6: Final Model
    final_features = (
        PROJECT_ROOT / "6_final_model" / "outputs" / cohort_name / age_band_fname / 
        f"{cohort_name}_{age_band_fname}_train_final_features_no_leakage.csv"
    )
    # Also check alternative location
    if not final_features.exists():
        final_features = (
            PROJECT_ROOT / "6b_final_model_selection" / "outputs" / cohort_name / age_band_fname /
            f"{cohort_name}_{age_band_fname}_train_final_features_no_leakage.csv"
        )
    status["step6_final_model"] = "DONE" if final_features.exists() else "PENDING"
    
    # Step 7: FFA
    ffa_output = PROJECT_ROOT / "7_ffa_analysis" / "outputs" / cohort_name / age_band_fname
    ffa_exists = ffa_output.exists() and any(ffa_output.iterdir())
    status["step7_ffa"] = "DONE" if ffa_exists else "PENDING"
    
    # Step 8: SHAP
    shap_output = PROJECT_ROOT / "8_shap_analysis" / "outputs" / cohort_name / age_band_fname
    shap_exists = shap_output.exists() and any(shap_output.iterdir())
    status["step8_shap"] = "DONE" if shap_exists else "PENDING"
    
    return status


def print_risk_model_status(cohort_name: str, age_band: str):
    """Print a formatted status table for risk model steps and prerequisites."""
    # Check prerequisites first
    prerequisites = check_feature_engineering_prerequisites(cohort_name, age_band)
    status = check_risk_model_status(cohort_name, age_band)
    
    print(f"\n{'='*80}")
    print(f"Risk Model Status: {cohort_name} / {age_band}")
    print(f"{'='*80}")
    
    # Prerequisites section
    print(f"\n{'Prerequisites (Feature Engineering)':<50} {'Status':<15}")
    print(f"{'-'*80}")
    all_prereqs_met = all(prerequisites.values())
    
    prereq_labels = {
        "bupar": "BupaR Features",
        "fpgrowth": "FP-Growth Features",
        "pgx": "PGx Features",
        "dtw": "DTW Features",
        "model_data": "Model Data (model_events.parquet)"
    }
    
    for key, label in prereq_labels.items():
        exists = prerequisites.get(key, False)
        icon = "✅" if exists else "❌"
        print(f"{label:<50} {icon} {'READY' if exists else 'MISSING':<15}")
    
    if not all_prereqs_met:
        print(f"\n⚠️  WARNING: Feature engineering prerequisites not met!")
        print(f"   Please complete feature engineering using pgx_cohort_feature_engineering.ipynb")
        print(f"   Missing: {', '.join([k for k, v in prerequisites.items() if not v])}")
        return
    
    # Risk model steps section
    print(f"\n{'Risk Model Steps':<50} {'Status':<15}")
    print(f"{'-'*80}")
    
    step_names = {
        "step6_final_model": "6. Final Model Training",
        "step7_ffa": "7. FFA Analysis",
        "step8_shap": "8. SHAP Analysis",
    }
    
    for step_key, step_label in step_names.items():
        stat = status.get(step_key, "UNKNOWN")
        icon = "✅" if stat == "DONE" else "⏳"
        print(f"{step_label:<50} {icon} {stat:<15}")
    
    print(f"{'='*80}\n")
    
    # Summary
    done_count = sum(1 for s in status.values() if s == "DONE")
    total_count = len(status)
    print(f"Progress: {done_count}/{total_count} steps complete ({100*done_count/total_count:.1f}%)")
    
    if done_count == total_count:
        print("\n✅ Risk model pipeline complete!")


# Check current status
print_risk_model_status(COHORT_NAME, AGE_BAND)

## Risk Model Pipeline Execution

Run individual steps or the full risk model pipeline. Each step checks for existing outputs and skips if already complete (idempotent).

In [ ]:
def run_risk_model_step(step_name: str, cohort_name: str, age_band: str, force: bool = False) -> bool:
    """
    Run a single risk model step.
    
    Args:
        step_name: One of '6', '7', '8'
        cohort_name: Cohort name (e.g., 'opioid_ed')
        age_band: Age band (e.g., '0-12')
        force: If True, rerun even if outputs exist
    
    Returns:
        True if step completed successfully, False otherwise
    """
    # Check prerequisites first
    prerequisites = check_feature_engineering_prerequisites(cohort_name, age_band)
    if not all(prerequisites.values()):
        missing = [k for k, v in prerequisites.items() if not v]
        print(f"❌ Prerequisites not met. Missing: {', '.join(missing)}")
        print(f"   Please complete feature engineering using pgx_cohort_feature_engineering.ipynb")
        return False
    
    age_band_fname = age_band.replace("-", "_")
    python_bin = sys.executable
    
    print(f"\n{'='*80}")
    print(f"Running Step {step_name}: {cohort_name} / {age_band}")
    print(f"{'='*80}\n")
    
    try:
        if step_name == "6":
            script = PROJECT_ROOT / "6b_final_model_selection" / "run_final_model.py"
            cmd = [str(python_bin), str(script), "--cohort", cohort_name, "--age_band", age_band]
            
        elif step_name == "7":
            script = PROJECT_ROOT / "7_ffa_analysis" / "run_full_ffa_analysis.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "8":
            script = PROJECT_ROOT / "8_shap_analysis" / "run_shap_analysis.py"
            cmd = [str(python_bin), str(script), "--cohort", cohort_name, "--age_band", age_band]
            
        else:
            print(f"❌ Unknown step: {step_name}")
            return False
        
        # Check if step already complete (unless force)
        if not force:
            status = check_risk_model_status(cohort_name, age_band)
            step_key = f"step{step_name}_final_model" if step_name == "6" else f"step{step_name}_ffa" if step_name == "7" else f"step{step_name}_shap"
            # Map step names to actual status keys
            step_key_map = {
                "6": "step6_final_model",
                "7": "step7_ffa",
                "8": "step8_shap"
            }
            step_key = step_key_map.get(step_name)
            if step_key and step_key in status and status[step_key] == "DONE":
                print(f"⏭️  Step {step_name} already complete. Use force=True to rerun.")
                return True
        
        # Run the command
        result = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
        if result.returncode == 0:
            print(f"✅ Step {step_name} completed successfully")
            return True
        else:
            print(f"❌ Step {step_name} failed with return code {result.returncode}")
            return False
                
    except Exception as e:
        print(f"❌ Error running step {step_name}: {e}")
        return False


def run_full_risk_model(cohort_name: str, age_band: str, start_from: Optional[str] = None, force: bool = False):
    """
    Run the full risk model pipeline from a given step (or from the beginning).
    
    Args:
        cohort_name: Cohort name
        age_band: Age band
        start_from: Step to start from (e.g., '6', '7'). If None, starts from Step 6.
        force: If True, rerun steps even if outputs exist
    """
    # Check prerequisites first
    prerequisites = check_feature_engineering_prerequisites(cohort_name, age_band)
    if not all(prerequisites.values()):
        missing = [k for k, v in prerequisites.items() if not v]
        print(f"❌ Prerequisites not met. Missing: {', '.join(missing)}")
        print(f"   Please complete feature engineering using pgx_cohort_feature_engineering.ipynb")
        return
    
    steps = ["6", "7", "8"]
    
    if start_from:
        try:
            start_idx = steps.index(start_from)
            steps = steps[start_idx:]
        except ValueError:
            print(f"❌ Unknown step: {start_from}. Available steps: {', '.join(steps)}")
            return
    
    print(f"\n{'='*80}")
    print(f"Running Full Risk Model Pipeline: {cohort_name} / {age_band}")
    if start_from:
        print(f"Starting from step: {start_from}")
    print(f"{'='*80}\n")
    
    for step in steps:
        success = run_risk_model_step(step, cohort_name, age_band, force=force)
        if not success:
            print(f"\n❌ Pipeline stopped at step {step}")
            print(f"Fix the error and rerun from step {step} using:")
            print(f"  run_full_risk_model('{cohort_name}', '{age_band}', start_from='{step}')")
            return
        
        # Update status after each step
        print_risk_model_status(cohort_name, age_band)
    
    print(f"\n{'='*80}")
    print(f"✅ Risk model pipeline completed for {cohort_name} / {age_band}")
    print(f"{'='*80}\n")


# Example: Run a single step
# run_risk_model_step("6", COHORT_NAME, AGE_BAND)

# Example: Run full risk model pipeline
# run_full_risk_model(COHORT_NAME, AGE_BAND)

## Batch Status Overview

Check risk model status across all cohorts and age bands.

In [ ]:
def get_all_cohort_combinations() -> List[Tuple[str, str]]:
    """Get all valid cohort/age band combinations."""
    combinations = []
    
    # Cohort 1: opioid_ed (age bands < 65)
    opioid_age_bands = ["0-12", "13-24", "25-44", "45-54", "55-64"]
    for ab in opioid_age_bands:
        combinations.append(("opioid_ed", ab))
    
    # Cohort 2: non_opioid_ed (age bands >= 65)
    non_opioid_age_bands = ["65-74", "75-84", "85-94"]
    for ab in non_opioid_age_bands:
        combinations.append(("non_opioid_ed", ab))
    
    return combinations


def print_batch_risk_model_status():
    """Print a comprehensive status table for all cohort/age band combinations."""
    combinations = get_all_cohort_combinations()
    
    print(f"\n{'='*100}")
    print(f"Batch Risk Model Status - All Cohorts")
    print(f"{'='*100}")
    print(f"{'Cohort':<25} {'Age Band':<12} {'6':<6} {'7':<6} {'8':<6}")
    print(f"{'-'*100}")
    
    step_keys = ["step6_final_model", "step7_ffa", "step8_shap"]
    step_labels = ["6", "7", "8"]
    
    for cohort_name, age_band in combinations:
        status = check_risk_model_status(cohort_name, age_band)
        row = f"{cohort_name:<25} {age_band:<12}"
        
        for step_key, step_label in zip(step_keys, step_labels):
            stat = status.get(step_key, "UNKNOWN")
            icon = "✅" if stat == "DONE" else "⏳"
            row += f"{icon:<6}"
        
        print(row)
    
    print(f"{'-'*100}")
    
    # Summary statistics
    total_combinations = len(combinations)
    total_steps = len(step_keys)
    total_cells = total_combinations * total_steps
    
    done_count = 0
    for cohort_name, age_band in combinations:
        status = check_risk_model_status(cohort_name, age_band)
        done_count += sum(1 for step_key in step_keys if status.get(step_key) == "DONE")
    
    print(f"\nSummary: {done_count}/{total_cells} step/cohort combinations complete ({100*done_count/total_cells:.1f}%)")
    print(f"{'='*100}\n")


# Print batch status
print_batch_risk_model_status()

## Parallel Cohort Execution

Run multiple cohorts in parallel using ProcessPoolExecutor (following the pattern from `7_update_codes.py` and parallelization guide).

**Configuration:**
- Uses `ProcessPoolExecutor` for multi-process parallelization (bypasses Python GIL)
- Worker count from environment variable `PGX_WORKERS_MEDICAL` or defaults to CPU count - 4
- Each worker runs the full risk model pipeline for one cohort/age band combination
- Progress tracking and error handling included
- Prerequisites are checked before execution

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import subprocess

def _run_risk_model_worker(
    project_root: str,
    cohort_name: str, 
    age_band: str, 
    start_from: Optional[str] = None, 
    force: bool = False
) -> Dict[str, Any]:
    """
    Worker function to run full risk model pipeline for a single cohort/age band.
    Uses subprocess to call pipeline scripts (following pattern from parallelization guide).
    
    Args:
        project_root: Path to project root (passed as string for pickling)
        cohort_name: Cohort name
        age_band: Age band
        start_from: Optional step to start from
        force: If True, rerun even if outputs exist
    
    Returns:
        Dictionary with results: {'cohort': str, 'age_band': str, 'success': bool, 'error': str or None}
    """
    import sys
    from pathlib import Path
    
    project_root = Path(project_root)
    python_bin = sys.executable
    
    # Define risk model steps in order
    steps = ["6", "7", "8"]
    
    # Filter steps if start_from is specified
    if start_from:
        try:
            start_idx = steps.index(start_from)
            steps = steps[start_idx:]
        except ValueError:
            return {
                'cohort': cohort_name,
                'age_band': age_band,
                'success': False,
                'error': f"Unknown step: {start_from}"
            }
    
    # Run each step sequentially for this cohort
    for step in steps:
        try:
            if step == "6":
                script = project_root / "6b_final_model_selection" / "run_final_model.py"
                cmd = [str(python_bin), str(script), "--cohort", cohort_name, "--age_band", age_band]
            elif step == "7":
                script = project_root / "7_ffa_analysis" / "run_full_ffa_analysis.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "8":
                script = project_root / "8_shap_analysis" / "run_shap_analysis.py"
                cmd = [str(python_bin), str(script), "--cohort", cohort_name, "--age_band", age_band]
            else:
                return {
                    'cohort': cohort_name,
                    'age_band': age_band,
                    'success': False,
                    'error': f"Unknown step: {step}"
                }
            
            # Run the command
            result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True)
            
            if result.returncode != 0:
                return {
                    'cohort': cohort_name,
                    'age_band': age_band,
                    'success': False,
                    'error': f"Step {step} failed with return code {result.returncode}: {result.stderr[:500]}"
                }
                
        except Exception as e:
            return {
                'cohort': cohort_name,
                'age_band': age_band,
                'success': False,
                'error': f"Step {step} exception: {str(e)}"
            }
    
    # All steps completed successfully
    return {
        'cohort': cohort_name,
        'age_band': age_band,
        'success': True,
        'error': None
    }


def run_risk_model_parallel(
    cohort_combinations: List[Tuple[str, str]],
    start_from: Optional[str] = None,
    force: bool = False,
    max_workers: Optional[int] = None
) -> Dict[str, Any]:
    """
    Run risk model pipeline for multiple cohort/age band combinations in parallel.
    
    Args:
        cohort_combinations: List of (cohort_name, age_band) tuples
        start_from: Optional step to start from (e.g., '6', '7')
        force: If True, rerun even if outputs exist
        max_workers: Maximum number of parallel workers (defaults to CPU count - 4)
    
    Returns:
        Dictionary with results for each cohort
    """
    # Check prerequisites for all cohorts first
    print(f"\n{'='*80}")
    print(f"Checking prerequisites for {len(cohort_combinations)} cohorts...")
    print(f"{'='*80}\n")
    
    missing_prereqs = []
    for cohort_name, age_band in cohort_combinations:
        prerequisites = check_feature_engineering_prerequisites(cohort_name, age_band)
        if not all(prerequisites.values()):
            missing = [k for k, v in prerequisites.items() if not v]
            missing_prereqs.append((cohort_name, age_band, missing))
            print(f"❌ {cohort_name} / {age_band}: Missing prerequisites: {', '.join(missing)}")
    
    if missing_prereqs:
        print(f"\n⚠️  WARNING: {len(missing_prereqs)} cohort(s) have missing prerequisites!")
        print(f"   Please complete feature engineering using pgx_cohort_feature_engineering.ipynb")
        print(f"   Aborting parallel execution.")
        return {}
    
    print(f"✅ All prerequisites met. Proceeding with parallel execution.\n")
    
    # Determine worker count (following parallelization guide pattern)
    if max_workers is None:
        # Use environment variable or default to CPU count - 4
        env_workers = os.getenv('PGX_WORKERS_MEDICAL')
        if env_workers and env_workers.isdigit() and int(env_workers) > 0:
            max_workers = int(env_workers)
        else:
            # Default: CPU count - 4 (leave cores for system/OS)
            max_workers = max(1, multiprocessing.cpu_count() - 4)
    
    print(f"\n{'='*80}")
    print(f"Running {len(cohort_combinations)} cohorts in parallel with {max_workers} workers")
    print(f"{'='*80}\n")
    
    results = {}
    completed = 0
    total = len(cohort_combinations)
    
    # Use ProcessPoolExecutor (following pattern from 7_update_codes.py and parallelization guide)
    # Pass project_root as string for pickling across processes
    project_root_str = str(PROJECT_ROOT)
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all jobs (pass project_root_str for worker function)
        future_to_cohort = {
            executor.submit(_run_risk_model_worker, project_root_str, cohort, age_band, start_from, force): (cohort, age_band)
            for cohort, age_band in cohort_combinations
        }
        
        print(f"Submitted {len(future_to_cohort)} jobs to {max_workers} workers\n")
        
        # Process completed jobs as they finish
        for future in as_completed(future_to_cohort):
            cohort, age_band = future_to_cohort[future]
            completed += 1
            
            try:
                result = future.result()
                results[f"{cohort}_{age_band}"] = result
                
                if result['success']:
                    print(f"✅ [{completed}/{total}] {cohort} / {age_band} - Completed successfully")
                else:
                    print(f"❌ [{completed}/{total}] {cohort} / {age_band} - Failed: {result.get('error', 'Unknown error')}")
                    
            except Exception as e:
                results[f"{cohort}_{age_band}"] = {
                    'cohort': cohort,
                    'age_band': age_band,
                    'success': False,
                    'error': str(e)
                }
                print(f"❌ [{completed}/{total}] {cohort} / {age_band} - Exception: {e}")
    
    # Summary
    print(f"\n{'='*80}")
    print(f"Parallel Execution Summary")
    print(f"{'='*80}")
    successful = sum(1 for r in results.values() if r.get('success', False))
    failed = len(results) - successful
    print(f"Total: {len(results)} cohorts")
    print(f"✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    print(f"{'='*80}\n")
    
    if successful == len(results):
        print("✅ All risk model training complete!")
    
    return results


# Example: Run all pending cohorts in parallel (optimal for EC2: 2 cohorts at a time)
# pending_cohorts = [
#     ("opioid_ed", "0-12"),
#     ("opioid_ed", "55-64"),
#     ("non_opioid_ed", "65-74"),
#     ("non_opioid_ed", "75-84"),
#     ("non_opioid_ed", "85-94"),
# ]
# results = run_risk_model_parallel(pending_cohorts, start_from="6", max_workers=2)